# Análise do ID3 para PopOut

## Objetivos da análise
Pretendemos analisar o algoritmo ID3 por nós desenvolvido em três frentes:
- Performance preditiva num dataset simples de teste (iris)
- Performance preditiva num dataset gerado pelo MCTS para o jogo PopOut
- Capacidade para jogar PopOut contra humanos, MCTS, e outras instâncias de si mesmo

> Nota:
Todos os ficheiros referidos neste notebook encontram-se na diretório decision_tree

## Implementação do algoritmo

O ID3 (Iterative Dichotomiser 3) é um algoritmo de aprendizagem supervisionada que constrói árvores de decisão de forma recursiva. A cada passo, escolhe o atributo que melhor separa os dados com base no **Ganho de Informação** (*Information Gain*), usando a **Entropia** como medida de impureza.

### Entropia

A entropia (metrics.py -> get_enthropy) mede o grau de desordem de um conjunto de dados. Para uma variável alvo $y$ com classes $c_1, c_2, \dots, c_n$, a entropia é:

$$
H(y) = -\sum_{i=1}^{n} p_i \log_2(p_i)
$$

onde $p_i$ é a proporção de exemplos da classe $c_i$ no conjunto.
Valores próximos de $0$ indicam conjuntos puros (uma só classe); valores próximos de $\log_2(n)$ indicam máxima dispersão.

### Ganho de Informação

O Ganho de Informação (metrics.py -> calculate_information_gain) mede a redução da entropia ao particionar os dados pelo atributo $X$:

$$
IG(y, X) = H(y) - \sum_{v \in \text{vals}(X)} \frac{|y_{X=v}|}{|y|} \cdot H(y_{X=v})
$$

O algoritmo seleciona o atributo (e, no caso numérico, o ponto de corte) que maximiza $IG$.

### Tratamento de atributos numéricos

O código testa pontos de corte a meio dos valores consecutivos ordenados (id3.py > _get_best_split):

$$
\text{possible\_splits} = \frac{\text{unique\_vals}[1:] + \text{unique\_vals}[:-1]}{2}
$$

Para cada split candidato, os dados são divididos em $X \leq split$ e $X > split$, calculando-se o IG. Atributos inteiros com 4 ou menos valores únicos são tratados como categóricos (auxiliar.py -> is_numeric).

### Critérios de paragem

A construção recursiva da árvore termina quando:

- O conjunto de dados está vazio (todas as *features* foram usadas);
- A profundidade máxima (`max_depth`) é atingida;
- O número de amostras é inferior a `min_samples`.

Nestes casos, o nó torna-se folha e devolve a classe maioritária do subconjunto.

### Previsão para o jogo PopOut

O método `predict_valid()` foi adaptado para o cenário de jogo: percorre a árvore até uma folha e, se a classe prevista não for uma jogada válida, sobe na árvore até encontrar um nó cuja classe maioritária seja válida. Se nenhum nó tiver uma classe válida, escolhe aleatoriamente.

### Parâmetros

| Parâmetro | Descrição |
|-----------|-----------|
| `max_depth` | Profundidade máxima da árvore (se igual a None, não existe profundidade máxima) |
| `min_samples` | Número mínimo de amostras para continuar a dividir |

## 1 - Performance preditiva no dataset iris
O principal objetivo deste teste é verificar o correto funcionamento do algoritmo, visto tratar-se de um dataset bastante simples

In [1]:
import sys
import os
from pathlib import Path
sys.path.append(os.path.abspath(".."))

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [2]:
df = pd.read_csv(Path("..", "data", "iris.csv"))
df.head()

,ID,sepallength,sepalwidth,petallength,petalwidth,class
0,1,5.1,3.5,1.4,0.2,Iris-setosa
1,2,4.9,3.0,1.4,0.2,Iris-setosa
2,3,4.7,3.2,1.3,0.2,Iris-setosa
3,4,4.6,3.1,1.5,0.2,Iris-setosa
4,5,5.0,3.6,1.4,0.2,Iris-setosa


In [11]:
X = df.drop("class", axis=1)
y=df["class"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

In [12]:
from decision_tree import id3
tree = id3.ID3() #no arguments, min_samples=10 and max_depth=None
tree.fit(X_train, y_train)

In [13]:
y_pred = tree.predict(X_test)
print("Accuracy: ", accuracy_score(y_test, y_pred))
#o carlos soares deve-nos um jantar

Accuracy:  1.0


O modelo acertou em todos os (50) casos de teste. Dado o tamanho reduzido do dataset e a sua simplicidade isto não constitui uma grande surpresa.

In [14]:
tree.print_tree()

└─ Root  --->  [Split: Feature ID]
    └─ If Feature ID < 100.50  --->  [Split: Feature petallength]
        └─ If Feature petallength < 2.45  --->  [Split: Feature sepallength]
            └─ If Feature sepallength < 4.35  --->  Predict: Iris-setosa
            └─ If Feature sepallength >= 4.35  --->  [Split: Feature sepalwidth]
                └─ If Feature sepalwidth < 2.60  --->  Predict: Iris-setosa
                └─ If Feature sepalwidth >= 2.60  --->  [Split: Feature petallength]
                    └─ If Feature petallength < 0.15  --->  Predict: Iris-setosa
                    └─ If Feature petallength >= 0.15  --->  Predict: Iris-setosa
        └─ If Feature petallength >= 2.45  --->  [Split: Feature sepallength]
            └─ If Feature sepallength < 4.95  --->  Predict: Iris-versicolor
            └─ If Feature sepallength >= 4.95  --->  [Split: Feature sepalwidth]
                └─ If Feature sepalwidth < 2.10  --->  Predict: Iris-versicolor
                └─ If Featur

Podemos ver que, apesar de não limitarmos a profundidade máxima da árvore, a sua complexidade não é muito elevada.